# Modulo 1 · Lezione 9 — Eccezioni

**Corso di Intelligenza Artificiale — Livello base**

---

## Cosa vedremo oggi?

| # | Blocco | Cosa imparerai |
|---|--------|----------------|
| 1 | Cos'è un'eccezione? | Errori a runtime e perché non bisogna temerli |
| 2 | `try` / `except` | Catturare e gestire gli errori |
| 3 | Tipi di eccezioni | Le più comuni e quando compaiono |
| 4 | `else` e `finally` | Codice che si esegue con o senza errore |
| 5 | `raise` | Lanciare eccezioni proprie |
| 6 | Eccezioni personalizzate | Creare le tue classi di errore |
| 7 | Buone pratiche | Cosa catturare e cosa no |
| 8 | Pratica | Validatore di dati |

> 💡 **Idea centrale:** nel mondo reale, i programmi falliscono — l'utente inserisce dati sbagliati, il file non esiste, la rete cade. Un'eccezione non è un fallimento del programmatore: è il modo in cui Python dice «è successo qualcosa di inaspettato». Il nostro compito è decidere cosa fare quando accade.

---
## 1. Cos'è un'eccezione?

Quando Python incontra un errore a runtime, **lancia un'eccezione**: interrompe il programma e mostra un messaggio di errore chiamato **traceback**.

```
Traceback (most recent call last):
  File "programma.py", line 3, in <module>
    risultato = 10 / 0
ZeroDivisionError: division by zero
```

Il traceback ha tre parti:
1. **Dove è successo** — file e numero di riga
2. **Quale riga ha causato l'errore** — il codice
3. **Che tipo di errore è** — il nome dell'eccezione e un messaggio

### Differenza tra errori di sintassi ed eccezioni

| Tipo | Quando compare | Esempio |
|------|---------------|--------|
| `SyntaxError` | Prima dell'esecuzione (Python non riesce a leggere il codice) | `if True` senza `:` |
| Eccezione | Durante l'esecuzione (il codice è valido ma qualcosa fallisce) | `10 / 0` |

Le eccezioni **si possono catturare e gestire**. Gli errori di sintassi no.

In [ ]:
# Decommenta ogni riga per vedere quale eccezione lancia

# 10 / 0                     # ZeroDivisionError
# int("ciao")                # ValueError
# lista = [1,2,3]; lista[9]  # IndexError
# d = {}; d["chiave"]        # KeyError
# None + 1                   # TypeError
# open("non_esiste.txt")     # FileNotFoundError
# variabile_che_non_esiste   # NameError

print("Esegui una riga alla volta per vedere il traceback")

---
## 2. `try` / `except` — Catturare eccezioni

La struttura `try/except` permette di «provare» a eseguire del codice e decidere cosa fare se fallisce, invece di far rompere il programma.

```python
try:
    # codice che potrebbe fallire
except TipoDiErrore:
    # cosa fare se quell'errore si verifica
```

**Flusso di esecuzione:**
```
Entra nel try
   │
   ├─── Se tutto va bene ──► esce dal try/except e continua
   │
   └─── Se si verifica un errore ──► salta all'except corrispondente
                                     │
                                     └─► continua dopo il try/except
```

In [ ]:
# Senza gestione degli errori — il programma si rompe
# risultato = int("abc")   # ValueError: interrompe il programma

# Con try/except — il programma continua
try:
    risultato = int("abc")
    print(f"Risultato: {risultato}")   # questa riga NON viene eseguita
except ValueError:
    print("Errore: impossibile convertire 'abc' in intero")

print("Il programma continua qui")

In [ ]:
# Accedere al messaggio di errore con 'as'
try:
    risultato = 10 / 0
except ZeroDivisionError as e:
    print(f"Tipo di errore:   {type(e).__name__}")
    print(f"Messaggio errore: {e}")
    print(f"Oggetto completo: {repr(e)}")

In [ ]:
# Catturare più tipi di errore
def converti_a_intero(valore):
    """Converte un valore in intero, gestendo i possibili errori."""
    try:
        return int(valore)
    except ValueError:
        print(f"ValueError: '{valore}' non è un numero valido")
        return None
    except TypeError:
        print(f"TypeError: impossibile convertire {type(valore).__name__} in intero")
        return None

print(converti_a_intero("42"))     # funziona
print(converti_a_intero("ciao"))   # ValueError
print(converti_a_intero(None))     # TypeError
print(converti_a_intero(3.7))      # funziona (tronca)

In [ ]:
# Catturare vari errori con una sola clausola (tupla)
def leggi_numero(testo):
    try:
        return float(testo)
    except (ValueError, TypeError) as e:
        print(f"Impossibile convertire: {e}")
        return None

print(leggi_numero("3.14"))
print(leggi_numero("abc"))
print(leggi_numero(None))

---
## 3. Gerarchia e tipi di eccezioni

Tutte le eccezioni in Python ereditano da `BaseException`. Quelle che userai nella pratica quotidiana ereditano da `Exception`.

```
BaseException
└── Exception
    ├── ValueError        → valore non corretto (int("abc"))
    ├── TypeError         → tipo non corretto ("testo" + 5)
    ├── KeyError          → chiave non esiste nel dict
    ├── IndexError        → indice fuori intervallo
    ├── AttributeError    → attributo non esiste (None.upper())
    ├── ZeroDivisionError → divisione per zero
    ├── FileNotFoundError → file non trovato
    ├── PermissionError   → senza permessi di accesso
    ├── ImportError       → modulo non trovato
    ├── NameError         → variabile non definita
    ├── OverflowError     → numero troppo grande
    └── RuntimeError      → errore generico di esecuzione
```

Catturare `Exception` cattura quasi ogni errore. Catturare `BaseException` cattura anche `KeyboardInterrupt` (Ctrl+C) e `SystemExit` — quasi mai lo vogliamo.

In [ ]:
# Catalogo di eccezioni comuni
casi = [
    (lambda: 1 / 0,              "ZeroDivisionError"),
    (lambda: int("ciao"),        "ValueError"),
    (lambda: "a" + 1,            "TypeError"),
    (lambda: {}["chiave"],       "KeyError"),
    (lambda: [][0],              "IndexError"),
    (lambda: None.upper(),       "AttributeError"),
    (lambda: open("___x___"),    "FileNotFoundError"),
]

for funzione, nome_atteso in casi:
    try:
        funzione()
    except Exception as e:
        tipo = type(e).__name__
        icona = "✅" if tipo == nome_atteso else "❌"
        print(f"{icona} {tipo:<22} → {e}")

---
## 4. `else` e `finally` — Controllo completo del flusso

La struttura completa di gestione delle eccezioni ha quattro parti:

```python
try:
    # codice che può fallire
except TipoErrore:
    # solo se si verifica quell'errore
else:
    # solo se NON si è verificato nessun errore
finally:
    # SEMPRE, ci sia stato o no un errore
```

| Blocco | Si esegue |
|--------|------------|
| `try` | Sempre (è il tentativo) |
| `except` | Solo se si è verificato l'errore indicato |
| `else` | Solo se **non** si è verificato nessun errore |
| `finally` | **Sempre**, con o senza errore — ideale per liberare risorse |

In [ ]:
def dividi(a, b):
    try:
        risultato = a / b
    except ZeroDivisionError:
        print("  [except]  Divisione per zero")
        risultato = None
    else:
        print(f"  [else]    Tutto bene, risultato = {risultato}")
    finally:
        print(f"  [finally] Si esegue sempre")
    return risultato

print("--- Caso 1: senza errore ---")
dividi(10, 2)

print("\n--- Caso 2: con errore ---")
dividi(10, 0)

In [ ]:
# finally è indispensabile per liberare risorse
# (anche se con 'with' questo avviene automaticamente per i file)

def elabora_file(percorso):
    file = None
    try:
        file = open(percorso, "r", encoding="utf-8")
        contenuto = file.read()
        righe = contenuto.splitlines()
        print(f"Lette {len(righe)} righe")
        return righe
    except FileNotFoundError:
        print(f"File non trovato: {percorso}")
        return []
    finally:
        if file:                # se è stato aperto, va chiuso
            file.close()
            print("File chiuso")

# Anche se con 'with' è più pulito:
def elabora_file_v2(percorso):
    try:
        with open(percorso, "r", encoding="utf-8") as f:
            return f.read().splitlines()
    except FileNotFoundError:
        print(f"File non trovato: {percorso}")
        return []

elabora_file("M1_C9_Excepciones.ipynb")   # esiste
print()
elabora_file("non_esiste.txt")             # non esiste

---
## 5. `raise` — Lanciare eccezioni

Oltre a catturare eccezioni, possiamo **lanciarle noi stessi** con `raise`. Questo è utile per:
- Validare i dati di input all'inizio di una funzione.
- Propagare un errore aggiungendo più contesto.
- Segnalare condizioni di business che non sono errori di Python ma della nostra logica.

```python
raise TipoDiErrore("messaggio descrittivo")
```

In [ ]:
def calcola_radice(numero):
    """Calcola la radice quadrata. Lancia ValueError se il numero è negativo."""
    if numero < 0:
        raise ValueError(f"Impossibile calcolare la radice di un numero negativo: {numero}")
    return numero ** 0.5

# Caso valido
print(calcola_radice(16))

# Caso non valido — catturiamo l'eccezione che abbiamo lanciato
try:
    print(calcola_radice(-4))
except ValueError as e:
    print(f"Errore catturato: {e}")

In [ ]:
# Rilanciare un'eccezione aggiungendo contesto
def carica_configurazione(percorso):
    import json
    try:
        with open(percorso, "r", encoding="utf-8") as f:
            return json.load(f)
    except FileNotFoundError:
        raise FileNotFoundError(
            f"File di configurazione non trovato: '{percorso}'. "
            "Assicurati che esista prima di avviare l'applicazione."
        )
    except json.JSONDecodeError as e:
        raise ValueError(
            f"Il file '{percorso}' non è un JSON valido. Errore alla riga {e.lineno}: {e.msg}"
        ) from e   # 'from e' concatena l'eccezione originale

try:
    config = carica_configurazione("non_esiste.json")
except FileNotFoundError as e:
    print(f"FileNotFoundError: {e}")

---
## 6. Eccezioni personalizzate

Puoi creare le tue classi di eccezione ereditando da `Exception`. Questo permette di:
- Dare nomi significativi agli errori del tuo dominio.
- Aggiungere attributi extra (campo che ha fallito, valore errato, ecc.).
- Catturare solo i tuoi errori senza intercettare quelli di Python.

```python
class MioErrore(Exception):
    pass
```

In [ ]:
# Eccezione semplice — solo il nome già dà contesto
class EtaNonValidaError(Exception):
    pass

class NomeVuotoError(Exception):
    pass

def registra_utente(nome, eta):
    if not nome or not nome.strip():
        raise NomeVuotoError("Il nome non può essere vuoto")
    if not isinstance(eta, int) or eta < 0 or eta > 150:
        raise EtaNonValidaError(f"Età non valida: {eta}. Deve essere un intero tra 0 e 150")
    return {"nome": nome.strip(), "eta": eta}

casi = [
    ("Anna", 28),
    ("",    25),
    ("Luigi", -5),
    ("Maria", 200)
]

for nome, eta in casi:
    try:
        utente = registra_utente(nome, eta)
        print(f"✅ Registrato: {utente}")
    except NomeVuotoError as e:
        print(f"❌ NomeVuotoError: {e}")
    except EtaNonValidaError as e:
        print(f"❌ EtaNonValidaError: {e}")

In [ ]:
# Eccezione arricchita con attributi extra
class ErroreValidazione(Exception):
    """Eccezione per errori di validazione dei dati."""
    def __init__(self, campo, valore, messaggio):
        self.campo     = campo
        self.valore    = valore
        self.messaggio = messaggio
        super().__init__(f"[{campo}] {messaggio} (valore ricevuto: {repr(valore)})")

def valida_email(email):
    if not isinstance(email, str):
        raise ErroreValidazione("email", email, "deve essere una stringa")
    if "@" not in email:
        raise ErroreValidazione("email", email, "deve contenere '@'")
    if email.count("@") > 1:
        raise ErroreValidazione("email", email, "può contenere solo una '@'")
    return email.lower().strip()

casi_email = ["anna@mail.com", "senza_chiocciola", 123, "due@@mail.com"]

for email in casi_email:
    try:
        risultato = valida_email(email)
        print(f"✅ Email valida: {risultato}")
    except ErroreValidazione as e:
        print(f"❌ {e}")
        print(f"   Campo: {e.campo}, Valore: {e.valore}")

### Gerarchia di eccezioni proprie

Nei progetti grandi è utile creare una gerarchia: un'eccezione base per il progetto e sottoclassi per casi specifici.

In [ ]:
# Gerarchia di eccezioni per un'app
class AppError(Exception):
    """Base per tutti gli errori dell'applicazione."""

class ErroreDati(AppError):
    """Errori relativi ai dati."""

class ErroreFile(AppError):
    """Errori relativi ai file."""

class ErroreFormatoCSV(ErroreDati):
    """Il CSV non ha il formato atteso."""

class ErroreColonnaMancante(ErroreDati):
    """Una colonna richiesta non è nel CSV."""

# Catturare per livello di specificità
def elabora(tipo_errore):
    if tipo_errore == "colonna":
        raise ErroreColonnaMancante("Manca la colonna 'prezzo'")
    elif tipo_errore == "formato":
        raise ErroreFormatoCSV("Il separatore non è una virgola")
    elif tipo_errore == "file":
        raise ErroreFile("Il file non esiste")

for tipo in ["colonna", "formato", "file"]:
    try:
        elabora(tipo)
    except ErroreColonnaMancante as e:
        print(f"Colonna mancante (specifico): {e}")
    except ErroreDati as e:
        print(f"Errore dati (generale): {e}")
    except AppError as e:
        print(f"Errore app (base): {e}")

---
## 7. Buone pratiche

### ✅ Cosa fare

```python
# 1. Cattura eccezioni specifiche, non Exception generico
try:
    valore = int(testo)
except ValueError:        # ← specifico
    valore = 0

# 2. Usa 'as e' per accedere al messaggio
except ValueError as e:
    print(f"Errore: {e}")

# 3. Cattura il più vicino possibile all'errore
# 4. Non lasciare except vuoti — fai sempre qualcosa
# 5. Usa finally per liberare risorse
```

### ❌ Cosa evitare

In [ ]:
# ❌ ANTIPATTERN 1: catturare tutto senza fare nulla (silenziare gli errori)
try:
    risultato = int("abc")
except Exception:
    pass   # ← l'errore sparisce in silenzio, molto pericoloso

# ✅ Minimo: loggare l'errore
try:
    risultato = int("abc")
except ValueError as e:
    print(f"WARN: conversione fallita — {e}")
    risultato = 0

# ❌ ANTIPATTERN 2: usare le eccezioni per il normale controllo di flusso
def e_intero_v1(testo):
    try:
        int(testo)
        return True
    except ValueError:
        return False

# ✅ Meglio: controllare prima con il metodo adeguato
def e_intero_v2(testo):
    return testo.strip().lstrip("-").isdigit()

for t in ["42", "abc", "-7", "3.14"]:
    print(f"'{t}': v1={e_intero_v1(t)}  v2={e_intero_v2(t)}")

### Il pattern EAFP vs LBYL

Python ha due filosofie di gestione degli errori:

| Stile | Nome | Descrizione | Quando |
|--------|--------|-------------|--------|
| **EAFP** | Easier to Ask Forgiveness than Permission | Prova l'operazione e cattura se fallisce | Quando il caso normale è il più frequente |
| **LBYL** | Look Before You Leap | Controlla prima di agire | Quando il controllo è economico e chiaro |

In [ ]:
dati = {"nome": "Anna", "eta": 28}

# LBYL — controlla prima
if "email" in dati:
    email = dati["email"]
else:
    email = "senza email"

# EAFP — prova e cattura
try:
    email = dati["email"]
except KeyError:
    email = "senza email"

# In Python: .get() è il modo più idiomatico per questo caso
email = dati.get("email", "senza email")

print(f"Email: {email}")

---
## 8. Pratica — Validatore di dati

Costruiamo un sistema di validazione robusto che usa eccezioni personalizzate per comunicare esattamente cosa è andato storto nei dati in input. È il tipo di codice che trovi in qualsiasi applicazione reale che riceve dati da utenti o file esterni.

---

### Preparazione — Eccezioni base del validatore

Definiamo la gerarchia di eccezioni che useremo in tutti gli esercizi.

In [ ]:
# Gerarchia di eccezioni del validatore
class ErroreValidazione(Exception):
    """Base per tutti gli errori di validazione."""
    def __init__(self, campo, valore, messaggio):
        self.campo     = campo
        self.valore    = valore
        self.messaggio = messaggio
        super().__init__(f"[{campo}] {messaggio}")

class ErroreTipo(ErroreValidazione):
    """Il valore ha un tipo non corretto."""

class ErroreIntervallo(ErroreValidazione):
    """Il valore è fuori dall'intervallo consentito."""

class ErroreFormato(ErroreValidazione):
    """Il valore non rispetta il formato atteso."""

class ErroreRichiesto(ErroreValidazione):
    """Un campo obbligatorio è vuoto o è None."""
    def __init__(self, campo):
        super().__init__(campo, None, "campo obbligatorio vuoto o assente")

print("✅ Gerarchia di eccezioni definita")

---
### Esercizio 1 — Validatori di campo singoli

**Consegna:** Scrivi funzioni che validino un campo e lancino l'eccezione appropriata se falliscono:

1. `valida_testo(campo, valore, min_len=1, max_len=None)` — non vuoto, lunghezza entro i limiti.
2. `valida_intero(campo, valore, minimo=None, massimo=None)` — convertibile in `int`, entro l'intervallo.
3. `valida_decimale(campo, valore, minimo=None, massimo=None)` — convertibile in `float`, entro l'intervallo.
4. `valida_email(campo, valore)` — contiene `@`, ha un dominio con un punto.

Ogni funzione deve restituire il valore già convertito nel tipo corretto se la validazione passa.

**Esempio:**
```python
valida_intero("eta", "28", minimo=0, massimo=150)  →  28
valida_intero("eta", "abc", minimo=0, massimo=150) →  lancia ErroreTipo
valida_intero("eta", "-5", minimo=0, massimo=150)  →  lancia ErroreIntervallo
```

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 1</summary>

```python
def valida_testo(campo, valore, min_len=1, max_len=None):
    """Valida che il valore sia una stringa non vuota entro la lunghezza indicata."""
    if not isinstance(valore, str):
        raise ErroreTipo(campo, valore, f"atteso str, arrivato {type(valore).__name__}")
    valore = valore.strip()
    if len(valore) < min_len:
        raise ErroreRichiesto(campo) if min_len == 1 else \
              ErroreIntervallo(campo, valore, f"minimo {min_len} caratteri, ne ha {len(valore)}")
    if max_len is not None and len(valore) > max_len:
        raise ErroreIntervallo(campo, valore, f"massimo {max_len} caratteri, ne ha {len(valore)}")
    return valore

def valida_intero(campo, valore, minimo=None, massimo=None):
    """Converte in int e valida l'intervallo."""
    try:
        valore_int = int(valore)
    except (ValueError, TypeError):
        raise ErroreTipo(campo, valore, f"impossibile convertire '{valore}' in intero")
    if minimo is not None and valore_int < minimo:
        raise ErroreIntervallo(campo, valore_int, f"minimo consentito: {minimo}")
    if massimo is not None and valore_int > massimo:
        raise ErroreIntervallo(campo, valore_int, f"massimo consentito: {massimo}")
    return valore_int

def valida_decimale(campo, valore, minimo=None, massimo=None):
    """Converte in float e valida l'intervallo."""
    try:
        valore_float = float(valore)
    except (ValueError, TypeError):
        raise ErroreTipo(campo, valore, f"impossibile convertire '{valore}' in decimale")
    if minimo is not None and valore_float < minimo:
        raise ErroreIntervallo(campo, valore_float, f"minimo consentito: {minimo}")
    if massimo is not None and valore_float > massimo:
        raise ErroreIntervallo(campo, valore_float, f"massimo consentito: {massimo}")
    return valore_float

def valida_email(campo, valore):
    """Valida che il valore abbia un formato email di base."""
    if not isinstance(valore, str):
        raise ErroreTipo(campo, valore, "atteso str")
    valore = valore.strip().lower()
    if not valore:
        raise ErroreRichiesto(campo)
    if valore.count("@") != 1:
        raise ErroreFormato(campo, valore, "deve contenere esattamente una '@'")
    locale, dominio = valore.split("@")
    if not locale:
        raise ErroreFormato(campo, valore, "manca la parte prima della '@'")
    if "." not in dominio or len(dominio) < 3:
        raise ErroreFormato(campo, valore, "dominio non valido (manca il punto o troppo corto)")
    return valore

# Prove
prove = [
    (valida_testo,     ("nome", "Anna"),             {}),
    (valida_testo,     ("nome", ""),                 {}),
    (valida_intero,    ("eta", "28"),      {"minimo": 0, "massimo": 150}),
    (valida_intero,    ("eta", "abc"),     {"minimo": 0, "massimo": 150}),
    (valida_intero,    ("eta", "-5"),      {"minimo": 0, "massimo": 150}),
    (valida_decimale,  ("prezzo", "19.99"), {"minimo": 0}),
    (valida_email,     ("email", "anna@mail.com"), {}),
    (valida_email,     ("email", "senza_chiocciola"),   {}),
]

for func, args, kwargs in prove:
    try:
        risultato = func(*args, **kwargs)
        print(f"✅ {func.__name__}({args[1]!r}) → {risultato!r}")
    except ErroreValidazione as e:
        print(f"❌ {type(e).__name__}: {e}")
```

**Punti chiave:**
- Ogni funzione fa una sola cosa e lancia l'eccezione più specifica possibile.
- Il `try/except (ValueError, TypeError)` in `valida_intero` cattura sia stringhe non numeriche sia `None`.
- Le funzioni restituiscono il valore **già convertito** — il codice chiamante non deve riconvertirlo.
- `raise ErroreTipo(...) from None` sopprime il contesto originale se non vogliamo concatenare le eccezioni.
</details>

---
### Esercizio 2 — Validatore di modulo completo

**Consegna:** Scrivi una funzione `valida_modulo(dati)` che riceve un dizionario con i dati di un utente e valida tutti i campi usando le funzioni dell'esercizio precedente.

Lo schema atteso:
```python
{
    "nome":    str,   # 2-50 caratteri
    "cognome": str,   # 2-50 caratteri
    "eta":     int,   # 0-120
    "email":   str,   # formato email
    "prezzo":  float  # > 0
}
```

La funzione deve:
- Validare **tutti** i campi, anche se qualcuno fallisce — raccoglie tutti gli errori.
- Restituire `{"ok": True, "dati": {...}}` se tutto è valido.
- Restituire `{"ok": False, "errori": [{"campo": ..., "messaggio": ...}]}` se ci sono errori.

**Esempio:**
```
{'ok': False, 'errori': [
    {'campo': 'nome', 'messaggio': 'campo obbligatorio vuoto o assente'},
    {'campo': 'eta',  'messaggio': 'massimo consentito: 120'}
]}
```

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 2</summary>

```python
def valida_modulo(dati):
    """Valida tutti i campi del modulo. Raccoglie tutti gli errori."""
    errori      = []
    dati_puliti = {}

    # Mappa delle validazioni: campo → funzione con i suoi argomenti
    validazioni = [
        ("nome",    valida_testo,     {"min_len": 2, "max_len": 50}),
        ("cognome", valida_testo,     {"min_len": 2, "max_len": 50}),
        ("eta",     valida_intero,    {"minimo": 0,  "massimo": 120}),
        ("email",   valida_email,     {}),
        ("prezzo",  valida_decimale,  {"minimo": 0.01}),
    ]

    for campo, funzione, kwargs in validazioni:
        valore = dati.get(campo)   # None se il campo non esiste
        try:
            dati_puliti[campo] = funzione(campo, valore, **kwargs)
        except ErroreValidazione as e:
            errori.append({"campo": e.campo, "tipo": type(e).__name__, "messaggio": e.messaggio})

    if errori:
        return {"ok": False, "errori": errori}
    return {"ok": True, "dati": dati_puliti}

# Prova 1: dati corretti
dati_ok = {
    "nome":    "Anna",
    "cognome": "Rossi",
    "eta":     "28",
    "email":   "anna@mail.com",
    "prezzo":  "99.99"
}
risultato = valida_modulo(dati_ok)
print("Caso 1 (valido):")
print(f"  ok: {risultato['ok']}")
if risultato['ok']:
    for k, v in risultato['dati'].items():
        print(f"  {k}: {v!r} ({type(v).__name__})")

# Prova 2: vari errori
dati_sbagliati = {
    "nome":    "",
    "cognome": "Rossi",
    "eta":     "200",
    "email":   "senza_chiocciola",
    "prezzo":  "-10"
}
risultato = valida_modulo(dati_sbagliati)
print("\nCaso 2 (con errori):")
print(f"  ok: {risultato['ok']}")
for err in risultato['errori']:
    print(f"  [{err['tipo']}] {err['campo']}: {err['messaggio']}")
```

**Punti chiave:**
- Validiamo **tutti** i campi anche se qualcuno fallisce — accumuliamo in `errori` invece di fermarci al primo. Così l'utente riceve tutti i problemi in una volta.
- La mappa `validazioni` separa la configurazione dalla logica — aggiungere un nuovo campo significa aggiungere una riga alla lista.
- `dati.get(campo)` restituisce `None` se il campo non esiste — le funzioni di validazione lo tratteranno come `ErroreRichiesto`.
- La funzione restituisce sempre la stessa struttura: `{"ok": bool, ...}` — facile da consumare per chi la chiama.
</details>

---
### Esercizio 3 — Validatore di CSV riga per riga

**Consegna:** Scrivi una funzione `valida_csv(percorso)` che legga un CSV e validi ogni riga. Usa il CSV `vendite.csv` della lezione precedente.

Deve:
1. Controllare che il file esista (`FileNotFoundError`).
2. Controllare che le colonne richieste siano presenti.
3. Validare ogni riga: `quantita` intero ≥ 1, `prezzo_unitario` decimale ≥ 0.
4. Restituire `{"valide": [...], "non_valide": [{"riga": N, "errori": [...]}]}`.

**Output atteso:**
```
Righe valide:     15
Righe non valide: 0
```

In [ ]:
# ✏️ IL TUO CODICE QUI
import csv
from pathlib import Path


<details>
<summary>💡 Vedi soluzione — Esercizio 3</summary>

```python
import csv
from pathlib import Path

COLONNE_RICHIESTE = {"data", "prodotto", "categoria", "quantita", "prezzo_unitario", "venditore"}

def valida_csv(percorso):
    """Legge e valida un CSV di vendite riga per riga."""
    percorso = Path(percorso)

    # Livello 1: il file esiste
    if not percorso.exists():
        raise FileNotFoundError(f"Il file non esiste: {percorso}")

    valide     = []
    non_valide = []

    try:
        with open(percorso, "r", encoding="utf-8") as f:
            lettore = csv.DictReader(f)

            # Livello 2: colonne richieste
            colonne_presenti = set(lettore.fieldnames or [])
            mancanti = COLONNE_RICHIESTE - colonne_presenti
            if mancanti:
                raise ErroreValidazione(
                    "intestazione", list(mancanti),
                    f"colonne richieste assenti: {sorted(mancanti)}"
                )

            # Livello 3: validare ogni riga
            for num_riga, riga in enumerate(lettore, start=2):  # start=2: la 1 è l'intestazione
                errori_riga = []

                for campo, funzione, kwargs in [
                    ("quantita",        valida_intero,    {"minimo": 1}),
                    ("prezzo_unitario", valida_decimale,  {"minimo": 0}),
                    ("prodotto",        valida_testo,     {"min_len": 1, "max_len": 100}),
                    ("venditore",       valida_testo,     {"min_len": 1}),
                ]:
                    try:
                        riga[campo] = funzione(campo, riga.get(campo), **kwargs)
                    except ErroreValidazione as e:
                        errori_riga.append({"campo": e.campo, "messaggio": e.messaggio})

                if errori_riga:
                    non_valide.append({"riga": num_riga, "dati": dict(riga), "errori": errori_riga})
                else:
                    riga["totale"] = round(riga["quantita"] * riga["prezzo_unitario"], 2)
                    valide.append(dict(riga))

    except UnicodeDecodeError:
        raise ValueError(f"Il file '{percorso}' non è codificato in UTF-8")

    return {"valide": valide, "non_valide": non_valide}


# Prova con il CSV della lezione precedente
percorso_csv = Path("file_lezione8") / "vendite.csv"

try:
    risultato = valida_csv(percorso_csv)
    print(f"Righe valide:     {len(risultato['valide'])}")
    print(f"Righe non valide: {len(risultato['non_valide'])}")
    if risultato['non_valide']:
        for nv in risultato['non_valide']:
            print(f"  Riga {nv['riga']}: {nv['errori']}")
except FileNotFoundError as e:
    print(f"File non trovato: {e}")
except ErroreValidazione as e:
    print(f"Errore di struttura: {e}")
```

**Punti chiave:**
- Tre livelli di validazione annidati: file → struttura → righe. Ogni livello lancia la propria eccezione.
- Accumuliamo errori per riga come nell'esercizio precedente — l'utente vede tutti i problemi.
- `start=2` in `enumerate` perché la riga 1 è l'intestazione — i numeri di riga corrispondono all'editor.
- `set(lettore.fieldnames or [])` gestisce il caso di file vuoto senza colonne.
</details>

---
### Esercizio 4 — Sistema di log degli errori

**Consegna:** Scrivi un decoratore `con_log_errori(funzione)` che avvolga qualsiasi funzione e, se lancia un'eccezione:
1. La registri in `errori.log` con timestamp, tipo di errore e messaggio.
2. Restituisca `None` invece di propagare l'errore.
3. Se la funzione funziona bene, restituisca il suo risultato normalmente.

**Esempio d'uso:**
```python
@con_log_errori
def dividi(a, b):
    return a / b

print(dividi(10, 2))   # 5.0
print(dividi(10, 0))   # None (errore registrato nel log)
```

In [ ]:
# ✏️ IL TUO CODICE QUI
from datetime import datetime


<details>
<summary>💡 Vedi soluzione — Esercizio 4</summary>

```python
from datetime import datetime
from pathlib import Path

LOG_PATH = Path("errori.log")

def con_log_errori(funzione):
    """Decoratore che cattura eccezioni, le registra nel log e restituisce None."""
    def wrapper(*args, **kwargs):
        try:
            return funzione(*args, **kwargs)
        except Exception as e:
            timestamp   = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            tipo_errore = type(e).__name__
            voce        = (
                f"[{timestamp}] [{tipo_errore}] "
                f"in {funzione.__name__}({args}, {kwargs}): {e}\n"
            )
            with open(LOG_PATH, "a", encoding="utf-8") as f:
                f.write(voce)
            print(f"⚠️  Errore registrato nel log: {tipo_errore}: {e}")
            return None
    return wrapper

# Applicare il decoratore
@con_log_errori
def dividi(a, b):
    return a / b

@con_log_errori
def cerca_in_dict(d, chiave):
    return d[chiave]

@con_log_errori
def converti(valore):
    return int(valore)

# Prove
print(dividi(10, 2))             # 5.0
print(dividi(10, 0))             # None + log
print(cerca_in_dict({"a": 1}, "a"))  # 1
print(cerca_in_dict({"a": 1}, "z"))  # None + log
print(converti("42"))            # 42
print(converti("abc"))           # None + log

# Vedere il log
print("\n--- Contenuto di errori.log ---")
if LOG_PATH.exists():
    print(LOG_PATH.read_text(encoding="utf-8"))
```

**Punti chiave:**
- Un **decoratore** è una funzione che riceve una funzione e ne restituisce un'altra migliorata — `@con_log_errori` è zucchero sintattico per `dividi = con_log_errori(dividi)`.
- `*args, **kwargs` in `wrapper` fa sì che funzioni con qualsiasi funzione, indipendentemente dai suoi parametri.
- Catturiamo `Exception` (non `BaseException`) — lasciamo passare `KeyboardInterrupt` e `SystemExit`.
- La modalità `"a"` nel log garantisce che ogni errore venga aggiunto senza cancellare i precedenti.
</details>

---
### Esercizio 5 (sfida) — Pipeline di validazione con report

**Consegna:** Integra tutto in una funzione `pipeline_validazione(percorso_csv, percorso_report)` che:

1. Provi a caricare il CSV — se fallisce, registra l'errore e termina con un report di fallimento.
2. Validi tutte le righe con `valida_csv`.
3. Elabori solo le righe valide (calcoli il totale di ciascuna).
4. Salvi le righe valide in un nuovo CSV `vendite_pulite.csv`.
5. Generi un report `.txt` con:
   - Totale delle righe elaborate, valide e non valide.
   - Dettaglio di ogni riga non valida e i suoi errori.
   - Somma totale delle righe valide.
6. Usi `finally` per garantire che il report venga **sempre** scritto anche in caso di errore.

In [ ]:
# ✏️ IL TUO CODICE QUI
import csv
import json
from pathlib import Path
from datetime import datetime


<details>
<summary>💡 Vedi soluzione — Esercizio 5</summary>

```python
import csv
from pathlib import Path
from datetime import datetime

def pipeline_validazione(percorso_csv, percorso_report):
    """Pipeline completo: carica → valida → salva → riporta."""
    percorso_csv    = Path(percorso_csv)
    percorso_report = Path(percorso_report)
    adesso          = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    righe_report = [
        "=" * 55,
        f"  REPORT DI VALIDAZIONE — {adesso}",
        "=" * 55,
        f"File: {percorso_csv.name}",
        ""
    ]
    risultato = None

    try:
        # Passo 1: validare il CSV
        risultato  = valida_csv(percorso_csv)
        valide     = risultato["valide"]
        non_valide = risultato["non_valide"]
        totale     = len(valide) + len(non_valide)

        # Passo 2: calcolare il totale delle righe valide
        somma_totale = sum(v["totale"] for v in valide)

        # Passo 3: salvare le righe valide
        percorso_pulito = percorso_csv.parent / "vendite_pulite.csv"
        if valide:
            with open(percorso_pulito, "w", encoding="utf-8", newline="") as f:
                scrittore = csv.DictWriter(f, fieldnames=valide[0].keys())
                scrittore.writeheader()
                scrittore.writerows(valide)

        # Passo 4: costruire il report
        righe_report += [
            f"Totale righe elaborate: {totale}",
            f"Righe valide:           {len(valide)}",
            f"Righe non valide:       {len(non_valide)}",
            f"Totale fatturato:       {somma_totale:,.2f} €",
            ""
        ]

        if non_valide:
            righe_report.append("--- RIGHE CON ERRORI ---")
            for nv in non_valide:
                righe_report.append(f"  Riga {nv['riga']}:")
                for err in nv["errori"]:
                    righe_report.append(f"    [{err['campo']}] {err['messaggio']}")
        else:
            righe_report.append("✅ Tutte le righe sono valide.")

        if valide:
            righe_report.append(f"\nCSV pulito salvato in: {percorso_pulito.name}")

    except FileNotFoundError as e:
        righe_report += [
            "❌ ERRORE CRITICO: file non trovato",
            str(e)
        ]
    except ErroreValidazione as e:
        righe_report += [
            "❌ ERRORE CRITICO: struttura del CSV non valida",
            str(e)
        ]
    except Exception as e:
        righe_report += [
            f"❌ ERRORE INASPETTATO: {type(e).__name__}",
            str(e)
        ]
    finally:
        # Il report viene SEMPRE scritto, con o senza errore
        righe_report += ["", "=" * 55]
        contenuto = "\n".join(righe_report)
        percorso_report.write_text(contenuto, encoding="utf-8")
        print(f"✅ Report salvato in {percorso_report}")
        print()
        print(contenuto)

# Eseguire il pipeline
pipeline_validazione(
    Path("file_lezione8") / "vendite.csv",
    Path("file_lezione8") / "report_validazione.txt"
)
```

**Punti chiave:**
- `finally` garantisce che il report venga scritto anche se c'è un errore in qualsiasi passo — fondamentale per pipeline di dati dove serve tracciabilità.
- Catturiamo le eccezioni dal più specifico al più generale: `FileNotFoundError` → `ErroreValidazione` → `Exception`. L'ordine conta.
- `risultato = None` prima del `try` permette a `finally` di sapere se il processo è arrivato al completamento.
- Accumulare le righe del report in una lista e scrivere alla fine (in `finally`) è più robusto che scrivere riga per riga.
</details>

---
## Riepilogo della lezione

### Struttura completa

```python
try:
    risultato = operazione_rischiosa()
except TipoSpecifico as e:      # cattura quel tipo concreto
    gestisci(e)
except (TypeError, ValueError): # cattura più tipi
    gestisci_insieme()
except Exception as e:          # cattura qualsiasi errore
    log(e)
else:
    usa(risultato)              # solo se non c'è stato errore
finally:
    libera_risorse()            # sempre
```

### Lanciare eccezioni

```python
raise ValueError("messaggio")        # lanciare
raise MioErrore(campo, valore, msg)  # eccezione propria
raise NuovoErrore("contesto") from e # concatenare
```

### Eccezioni personalizzate

```python
class MioErrore(Exception):
    def __init__(self, campo, valore, messaggio):
        self.campo  = campo
        self.valore = valore
        super().__init__(f"[{campo}] {messaggio}")
```

### Regole d'oro

| Regola | Descrizione |
|-------|-------------|
| Specifico > generale | Cattura `ValueError` prima di `Exception` |
| Non silenziare mai | Un `except: pass` nasconde bug |
| `finally` per le risorse | Garantisce la chiusura di file, connessioni… |
| `raise` con contesto | Aggiunge informazioni quando rilanci |
| Valida ai confini | Input utente, file, API |

---

## Cosa vedremo nella prossima lezione?

Nella **Lezione 10** impareremo la **Programmazione Orientata agli Oggetti (POO)**:
- Classi e oggetti
- Attributi e metodi
- Ereditarietà

Le eccezioni personalizzate che abbiamo creato oggi usano già l'ereditarietà — nella prossima lezione capiremo esattamente come funziona questo meccanismo.